# YouTube 업로드 주기와 예약 작업 간격 분석

2026-08-31 운영 공개 API의 최근 업로드를 기준으로, WebSub 누락 복구용 channel reconcile과 관련 유지보수 작업의 적정 주기를 검토한다. 운영 D1 직접 조회는 Cloudflare API 7403 권한 오류로 불가능해 공개 API를 대체 근거로 사용한다.

In [ ]:
from datetime import datetime
from statistics import median
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json

BASE = 'https://otw-schedule.info'

def fetch_json(path, params=None):
    query = ('?' + urlencode(params)) if params else ''
    request = Request(BASE + path + query, headers={'User-Agent': 'otw-schedule-cadence-audit/1.0'})
    with urlopen(request, timeout=20) as response:
        return json.load(response)

def percentile(values, p):
    if not values:
        return None
    ordered = sorted(values)
    index = (len(ordered) - 1) * p
    low, high = int(index), min(int(index) + 1, len(ordered) - 1)
    return ordered[low] + (ordered[high] - ordered[low]) * (index - low)

def rolling_max(times, hours):
    best = 0
    for start, first in enumerate(times):
        count = 0
        for current in times[start:]:
            if (current - first).total_seconds() <= hours * 3600:
                count += 1
            else:
                break
        best = max(best, count)
    return best

members = fetch_json('/api/members')
channel_ids = [row['youtube_channel_id'] for row in members if row.get('youtube_channel_id')]
official = fetch_json('/api/youtube/videos', {'channelIds': ','.join(channel_ids), 'maxResults': 20})
kirinuki = fetch_json('/api/kirinuki/videos', {'maxResults': 40})
official_names = {row['youtube_channel_id']: row['name'] for row in members}

def summarize(row, source):
    content = row['content']
    items = {item['videoId']: item for item in content.get('videos', []) + content.get('shorts', [])}
    times = sorted(datetime.fromisoformat(item['publishedAt'].replace('Z', '+00:00')) for item in items.values())
    gaps = [(right - left).total_seconds() / 3600 for left, right in zip(times, times[1:])]
    span_days = ((times[-1] - times[0]).total_seconds() / 86400) if len(times) > 1 else 0
    return {
        'source': source,
        'channel': official_names.get(row['channelId'], row.get('channelName', row['channelId'])),
        'observed_uploads': len(times),
        'window_days': round(span_days, 1),
        'uploads_per_week': round((len(times) - 1) / span_days * 7, 2) if span_days else 0,
        'p10_gap_hours': round(percentile(gaps, 0.10), 2) if gaps else None,
        'median_gap_hours': round(median(gaps), 2) if gaps else None,
        'max_uploads_4h': rolling_max(times, 4),
        'max_uploads_24h': rolling_max(times, 24),
    }

rows = [summarize(row, 'official') for row in official['byChannel']]
rows += [summarize(row, 'kirinuki') for row in kirinuki['byChannel']]
rows.sort(key=lambda row: (row['source'], -row['uploads_per_week']))
print(json.dumps({
    'measured_at': datetime.now().astimezone().isoformat(),
    'channel_count': len(rows),
    'observed_uploads': sum(row['observed_uploads'] for row in rows),
    'busiest_4h': max(row['max_uploads_4h'] for row in rows),
    'busiest_24h': max(row['max_uploads_24h'] for row in rows),
    'rows': rows,
}, ensure_ascii=False, indent=2))

In [ ]:
scenarios = [
    {'probe': '15분', 'runs_per_day': 96},
    {'probe': '1시간', 'runs_per_day': 24},
    {'probe': '4시간', 'runs_per_day': 6},
]
for scenario in scenarios:
    scenario['reduction_vs_15m_pct'] = round((1 - scenario['runs_per_day'] / 96) * 100, 1)
print(json.dumps(scenarios, ensure_ascii=False, indent=2))

## 2026-08-31 관측 결과와 결정

- 공식 8개·키리누키 6개 채널에서 최근 업로드 373건을 관측했다.
- 가장 바쁜 채널은 주 9.11건, 중앙 업로드 간격 19.49시간이었다. 4시간 창의 최대 업로드는 3건, 24시간 창은 5건이었다.
- 공식 채널 중 가장 바쁜 채널도 주 2.08건, 중앙 업로드 간격 76.35시간이었다.
- 업로드 playlist 한 페이지 50건보다 관측 최대 밀도가 크게 낮아 4시간 수집은 충분히 안전하다. 다만 WebSub가 즉시 알림을 담당하고 기존 채널별 reconcile 간격이 이미 6시간이므로 이를 4시간으로 당겨 외부 호출을 늘릴 근거는 없다.
- 최종안은 채널별 실제 reconcile 6시간 유지, 빈 due 확인은 15분에서 1시간으로 축소한다. ingestion recovery와 WebSub maintenance도 1시간, source health due 확인도 1시간으로 완화한다.

### 한계

공개 API의 canonical 최근 구간(공식 채널당 최대 20건, 키리누키 채널당 최대 40건)을 사용한 단면 분석이다. 장기 계절성은 포함하지 않지만, 누락 위험 판단에는 최근 최고 업로드 밀도와 WebSub 1차 경로를 함께 반영했다.